# nb48 - Masked-cell pretraining (H10)

**Error analysis.** With only ~51k labeled minbias training windows the encoder must learn shower shape, pileup texture, and calibration simultaneously; the campaign shows supervision, not architecture, binds. The one family never tested here is self-supervised pretraining, and it does not need Etrue at all.

**Question.** Does masked-cell pretraining on a larger unlabeled window set improve the fine-tuned resolution at our label scale?

**Hypothesis.** H10: pretrain the encoder to reconstruct the energies of masked cells from context (learning shower/pileup structure), then fine-tune the quant model from that initialization; published evidence shows large gains exactly at 10^4-scale fine-tuning.

**Research.** Tokenizer-free masked particle modeling: Leigh et al., MLST 2025, arXiv:2409.12589; pretraining-objectives study (MPM+supervised strongest in low-label regime): arXiv:2606.14870; tau FM regression ~50% resolution gain at 10^4 events: Tani, Pata, Birk, arXiv:2503.19165.

**Proof criterion.** Fine-tune = exact nb43 quant recipe (pure minbias, W=4, 2 seeds); anchors 0.0445 +/- 0.0001 singles / ens 0.0437. Win = >0.002 overall or in any E>17 GeV bin. Pretraining uses relaxed-selection windows with NO Etrue in the objective.

In [1]:
import os, sys, copy, time, pathlib
import numpy as np, pandas as pd
import torch, torch.nn as nn
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import split, resolution, PITCH, EPS
from picocal_data import build_grid, make_windows, splits_for, prep, THRESH, NC
OUT = REPO / 'reports' / 'predictions'
CKPT = REPO / '.scratch' / 'ckpt'; CKPT.mkdir(parents=True, exist_ok=True)
DEVICE = os.environ.get('NB48_DEVICE') or ('cuda' if torch.cuda.is_available() else 'cpu')
MODE = os.environ.get('NB48_MODE', 'full')
MBF = sorted((REPO / 'data' / 'minimum_bias').glob('*.root'))
if MODE == 'smoke': MBF = MBF[:8]
t0 = time.time()
ME = build_grid(MBF, 'minbias labeled')
MU = build_grid(MBF, 'minbias relaxed', vertex_max=1e9, emin=0.05, emax=1e9)
D = prep(4, ME, None, ng=5)
print(f'device {DEVICE} | mode {MODE} | build+prep {time.time()-t0:.0f}s')

minbias labeled: 72554 events


minbias relaxed: 80901 events


W=4: N 72554 (main 72554 + aux 0), tr/va/te 50787/10883/10884, IN_DIM 16
device cuda | mode full | build+prep 213s


In [2]:
W = 4
urows, _ = make_windows(W, MU)
L = (2*W+1)**2; IN_DIM = D['IN_DIM']
NU = len(urows)
XU = np.zeros((NU, L, IN_DIM), np.float32); MUK = np.zeros((NU, L), np.bool_)
for i, (tok, se, sde, et, rg, etv) in enumerate(urows):
    n = tok.shape[0]; XU[i, :n] = tok; MUK[i, :n] = True
XU[:, :, :NC] = (XU[:, :, :NC] - D['mean']) / D['std']; XU[~MUK] = 0.0
TU = dict(X=torch.from_numpy(XU).to(DEVICE), M=torch.from_numpy(MUK).to(DEVICE))
print(f'pretrain windows: {NU} (labeled set: {len(D["y"])})')

pretrain windows: 80901 (labeled set: 72554)


In [3]:
CFG = dict(d=128, nhead=4, layers=3, dropout=0.1, lr=3e-4, wd=1e-4, batch=96)
def build_encoder():
    d = CFG['d']
    embed = nn.Linear(IN_DIM, d)
    layer = nn.TransformerEncoderLayer(d, CFG['nhead'], dim_feedforward=4*d,
                                       dropout=CFG['dropout'], batch_first=True)
    enc = nn.TransformerEncoder(layer, CFG['layers'], enable_nested_tensor=False)
    return embed, enc
class MPMNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed, self.enc = build_encoder()
        self.mask_emb = nn.Parameter(torch.zeros(CFG['d']))
        self.dec = nn.Sequential(nn.Linear(CFG['d'], CFG['d'] // 2), nn.ReLU(), nn.Linear(CFG['d'] // 2, 1))
    def forward(self, x, m, mk):
        h = self.embed(x)
        h = torch.where(mk.unsqueeze(-1), self.mask_emb.expand_as(h), h)
        h = self.enc(h, src_key_padding_mask=~m)
        return self.dec(h).squeeze(-1)
PEP = {'smoke': 1, 'full': 30}[MODE]
MRATE = 0.25
def pretrain():
    torch.manual_seed(100); rng = np.random.default_rng(100)
    net = MPMNet().to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=PEP)
    ck = CKPT / 'nb48_mpm.pt'
    ep0 = 0
    if ck.exists():
        st = torch.load(ck, map_location=DEVICE)
        net.load_state_dict(st['model']); opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
        ep0 = st['ep'] + 1
        rng = np.random.default_rng(100 + 1000 * ep0)
        print(f'  resume pretrain from epoch {ep0}', flush=True)
    idxall = np.arange(NU)
    for ep in range(ep0, PEP):
        net.train(); tot = 0.0; k = 0
        for j in range(0, NU, CFG['batch']):
            b = torch.from_numpy(rng.permutation(idxall)[j:j+CFG['batch']] if j == 0 else idxall[j:j+CFG['batch']]).to(DEVICE)
            xb, mb = TU['X'][b], TU['M'][b]
            mk = (torch.rand_like(mb, dtype=torch.float32) < MRATE) & mb
            if not mk.any(): continue
            opt.zero_grad()
            pred = net(xb, mb, mk)
            loss = nn.functional.mse_loss(pred[mk], xb[:, :, 0][mk])
            loss.backward(); opt.step()
            tot += loss.item(); k += 1
        sched.step()
        torch.save(dict(model=net.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), ep=ep), ck)
        print(f'  pretrain ep {ep}: masked-cell mse {tot/max(k,1):.4f}', flush=True)
    return net
t1 = time.time()
mpm = pretrain()
print(f'pretrain done {time.time()-t1:.0f}s')

  pretrain ep 0: masked-cell mse 0.7540


  pretrain ep 1: masked-cell mse 0.7411


  pretrain ep 2: masked-cell mse 0.7389


  pretrain ep 3: masked-cell mse 0.7378


  pretrain ep 4: masked-cell mse 0.7370


...

  pretrain ep 26: masked-cell mse 0.7275


  pretrain ep 27: masked-cell mse 0.7266


  pretrain ep 28: masked-cell mse 0.7268


  pretrain ep 29: masked-cell mse 0.7273


pretrain done 483s


In [4]:
NG = 5
class SubNetFT(nn.Module):
    def __init__(self, la0, lb0):
        super().__init__()
        self.embed, self.enc = build_encoder()
        d = CFG['d']
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d + NG, d), nn.ReLU(), nn.Dropout(CFG['dropout']), nn.Linear(d, 3))
        self.fhead = nn.Sequential(nn.Linear(d, d // 2), nn.ReLU(), nn.Linear(d // 2, 1))
        self.la = nn.Parameter(torch.tensor(float(la0))); self.lb = nn.Parameter(torch.tensor(float(lb0)))
    def forward(self, x, m, g, ecell):
        h = self.enc(self.embed(x), src_key_padding_mask=~m)
        w = torch.sigmoid(self.fhead(h).squeeze(-1)) * m.float()
        base = self.la * torch.log1p((w * ecell).sum(1, keepdim=True)) + self.lb
        wm = m.unsqueeze(-1).float()
        p = self.norm((h * wm).sum(1) / wm.sum(1).clamp(min=1))
        return base + self.head(torch.cat([p, g], 1))
T = dict(X=torch.from_numpy(D['X']).to(DEVICE), M=torch.from_numpy(D['M']).to(DEVICE),
         G=torch.from_numpy(D['G']).to(DEVICE), Y=torch.from_numpy(D['y']).unsqueeze(1).to(DEVICE),
         E=torch.from_numpy(D['Eraw']).to(DEVICE))
ktr, kva, kte = D['ktr'], D['kva'], D['kte']
y = D['y']; Et = D['Et']
QS = torch.tensor([0.25, 0.5, 0.75], device=DEVICE)
def pinball(q, yb):
    d = yb - q
    return torch.maximum(QS * d, (QS - 1) * d).mean()
def wcalib(qv, qt, yva):
    wv = qv[:, 2] - qv[:, 0]; wt_ = qt[:, 2] - qt[:, 0]
    cuts = np.quantile(wv, [1/3, 2/3])
    gv = np.digitize(wv, cuts); gt = np.digitize(wt_, cuts)
    pe = np.empty(len(qt))
    for g in range(3):
        if (gv == g).sum() < 10 or (gt == g).sum() == 0:
            a, b2 = np.polyfit(qv[:, 1], yva, 1)
        else:
            a, b2 = np.polyfit(qv[gv == g, 1], yva[gv == g], 1)
        pe[gt == g] = np.exp(a * qt[gt == g, 1] + b2)
    return pe
def finetune(seed, epochs, patience):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = SubNetFT(D['la0'], D['lb0']).to(DEVICE)
    model.embed.load_state_dict(mpm.embed.state_dict())
    model.enc.load_state_dict(mpm.enc.state_dict())
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    ck = CKPT / f'nb48_pre_s{seed}.pt'
    def batches(idx, bs, sh):
        idx = np.asarray(idx)
        if sh: idx = rng.permutation(idx)
        for j in range(0, len(idx), bs): yield torch.from_numpy(idx[j:j+bs]).to(DEVICE)
    def fwd(b): return model(T['X'][b], T['M'][b], T['G'][b], T['E'][b])
    def run(idx):
        model.eval(); out = []
        with torch.no_grad():
            for b in batches(idx, 256, False): out.append(fwd(b).cpu().numpy())
        return np.concatenate(out)
    def vloss():
        model.eval(); s = 0.0; k = 0
        with torch.no_grad():
            for b in batches(kva, 256, False):
                s += pinball(fwd(b), T['Y'][b]).item(); k += 1
        return s / max(k, 1)
    best = 1e9; bstate = None; wait = 0; ep0 = 0
    if ck.exists():
        st = torch.load(ck, map_location=DEVICE)
        model.load_state_dict(st['model']); opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
        best = st['best']; bstate = st['bstate']; wait = st['wait']; ep0 = st['ep'] + 1
        rng = np.random.default_rng(seed + 1000 * ep0)
        print(f'  resume s{seed} from epoch {ep0}', flush=True)
    for ep in range(ep0, epochs):
        model.train()
        for b in batches(ktr, CFG['batch'], True):
            opt.zero_grad()
            pinball(fwd(b), T['Y'][b]).backward()
            opt.step()
        sched.step(); vv = vloss()
        if vv < best - 1e-4: best = vv; bstate = copy.deepcopy(model.state_dict()); wait = 0
        else: wait += 1
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                        best=best, bstate=bstate, wait=wait, ep=ep), ck)
        if wait >= patience: break
    model.load_state_dict(bstate)
    pe = wcalib(run(kva), run(kte), y[kva])
    return float(resolution(pe, Et[kte])['sigma_eff']), pe

In [5]:
EPOCHS = {'smoke': 2, 'full': 100}[MODE]
PATIENCE = {'smoke': 99, 'full': 15}[MODE]
SEEDS = {'smoke': [0], 'full': [0, 1]}[MODE]
TAG = '' if MODE == 'full' else '_smoke'
CSVP = OUT / f'nb48_pretrain{TAG}.csv'
done = set()
if CSVP.exists():
    done = set(pd.read_csv(CSVP)['seed'])
    print('resume, done:', sorted(done))
for seed in SEEDS:
    if seed in done: print('skip', seed); continue
    t1 = time.time()
    sig, pe = finetune(seed, EPOCHS, PATIENCE)
    np.save(OUT / f'nb48_pred{TAG}_pre_s{seed}.npy', pe)
    row = dict(seed=seed, sigma_eff=round(sig, 4), elapsed=round(time.time()-t1))
    pd.DataFrame([row]).to_csv(CSVP, mode='a', header=not CSVP.exists() or CSVP.stat().st_size == 0, index=False)
    print(f'pre seed {seed}: sigma_eff {sig:.4f} ({row["elapsed"]}s)', flush=True)
print(pd.read_csv(CSVP).to_string(index=False))

pre seed 0: sigma_eff 0.0463 (767s)


pre seed 1: sigma_eff 0.0442 (918s)


 seed  sigma_eff  elapsed
    0     0.0463      767
    1     0.0442      918


## Verdict vs nb43 quant anchor

Same recipe, same splits; the only change is the pretrained initialization. Win = >0.002 overall or in any E>17 bin.

In [6]:
te_e = Et[kte]
edges = np.quantile(te_e, np.linspace(0, 1, 7))
def perbin(pe):
    out = []
    for i in range(6):
        hi = edges[i+1] + (1e-9 if i == 5 else 0)
        mm = (te_e >= edges[i]) & (te_e < hi)
        out.append(resolution(pe[mm], te_e[mm])['sigma_eff'])
    return out
print('anchor nb43 quant: 0.0445 +/- 0.0001 | ens 0.0437 | per-bin 0.0659/0.0479/0.0376/0.0362/0.0344/0.0387')
preds = [np.load(OUT / f'nb48_pred{TAG}_pre_s{s}.npy') for s in SEEDS
         if (OUT / f'nb48_pred{TAG}_pre_s{s}.npy').exists()]
if preds:
    sig = [resolution(p, te_e)['sigma_eff'] for p in preds]
    ens = np.stack(preds).mean(0)
    print(f'pretrained mean {np.mean(sig):.4f} +/- {np.std(sig):.4f} | ens {resolution(ens, te_e)["sigma_eff"]:.4f}')
    print('per-bin ' + ' / '.join(f'{b:.4f}' for b in perbin(ens)))

anchor nb43 quant: 0.0445 +/- 0.0001 | ens 0.0437 | per-bin 0.0659/0.0479/0.0376/0.0362/0.0344/0.0387
pretrained mean 0.0452 +/- 0.0010 | ens 0.0439
per-bin 0.0686 / 0.0489 / 0.0374 / 0.0367 / 0.0351 / 0.0390
